<a href="https://colab.research.google.com/github/kinemax-core/Genomic-data-science-R/blob/main/human_chromosome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def naive_with_counts(p, t):
    num_alignments = 0
    num_comparisons = 0
    occurrences = []

    for i in range(len(t) - len(p) + 1):
        num_alignments += 1
        match = True
        for j in range(len(p)):
            num_comparisons += 1
            if t[i + j] != p[j]:
                match = False
                break
        if match:
            occurrences.append(i)

    return occurrences, num_alignments, num_comparisons

def read_fasta_completely_cleaned(filename):
    """Parses a FASTA file, stripping the header and removing ALL internal newline/spacing formatting."""
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            if not line.startswith('>'):
                # .strip() only removes trailing outer whitespace.
                # .replace() is REQUIRED to clear out hidden internal row line breaks (\n and \r)
                clean_line = line.strip().replace('\r', '').replace('\n', '')
                sequence.append(clean_line)
    return ''.join(sequence)

# --- Compute alignments tried ---
filename = 'chr1.GRCh38.excerpt.fasta'
pattern = "GGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGG"

t = read_fasta_completely_cleaned(filename)
p_len = len(pattern)
alignments_tried = len(t) - p_len + 1

print(f"Verified True Text Length (|T|): {len(t)} (Should be exactly 568214)")
print(f"Pattern Length (|P|): {p_len}")
print(f"Actual Alignments Tried: {alignments_tried}")

In [ ]:
def build_bad_character_table(p):
    """Generates the bad character lookup table."""
    table = {}
    for i in range(len(p) - 1):
        table[p[i]] = len(p) - 1 - i
    return table

def build_good_suffix_table(p):
    """Generates the good suffix shift table."""
    m = len(p)
    good_suffix = [m] * (m + 1)
    suff = [0] * m

    # Calculate suffix lengths
    suff[m - 1] = m
    g = m - 1
    f = 0
    for i in range(m - 2, -1, -1):
        if i > g and suff[i + m - 1 - f] < i - g:
            suff[i] = suff[i + m - 1 - f]
        else:
            if i < g:
                g = i
            f = i
            while g >= 0 and p[g] == p[g + m - 1 - f]:
                g -= 1
            suff[i] = f - g

    # Case 1: Substring matches suffix
    j = 0
    for i in range(m - 1, -1, -1):
        if suff[i] == i + 1:
            while j < m - 1 - i:
                if good_suffix[j] == m:
                    good_suffix[j] = m - 1 - i
                j += 1

    # Case 2: Suffix matches prefix
    for i in range(m - 1):
        good_suffix[m - 1 - suff[i]] = m - 1 - i

    return good_suffix

def boyer_moore_pure_counts(p, t):
    """Boyer-Moore algorithm tracking alignments and character comparisons."""
    num_alignments = 0
    num_comparisons = 0
    occurrences = []

    m = len(p)
    n = len(t)

    # Build tables on the fly
    bad_char = build_bad_character_table(p)
    good_suffix = build_good_suffix_table(p)

    i = 0
    while i <= n - m:
        num_alignments += 1
        mismatch = False

        # Scan from right to left
        for j in range(m - 1, -1, -1):
            num_comparisons += 1
            if p[j] != t[i + j]:
                mismatch = True

                # Bad character rule shift calculation
                bad_char_shift = bad_char.get(t[i + j], m) - (m - 1 - j)
                if bad_char_shift < 1:
                    bad_char_shift = 1

                # Good suffix rule shift calculation
                good_suffix_shift = good_suffix[j]

                i += max(bad_char_shift, good_suffix_shift)
                break

        if not mismatch:
            occurrences.append(i)
            i += good_suffix[0]  # Match skip shift

    return occurrences, num_alignments, num_comparisons

def read_fasta_cleaned(filename):
    """Parses a FASTA file, removing headers and all internal line breaks."""
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            if not line.startswith('>'):
                sequence.append(line.strip().replace('\r', '').replace('\n', ''))
    return ''.join(sequence)

# --- Execution ---

# 1. Load your actual file text
t = read_fasta_cleaned('chr1.GRCh38.excerpt.fasta')

# 2. Define your target pattern
pattern = "GGCGCGGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGG"

# 3. Calculate metrics
matches, alignments, comparisons = boyer_moore_pure_counts(pattern, t)

print(f"Total Text Length Checked: {len(t)} bases")
print(f"Matches found at indices: {matches}")
print(f"Boyer-Moore Alignments tried: {alignments}")
print(f"Boyer-Moore Character Comparisons made: {comparisons}")

In [ ]:
import bisect

class Index(object):
    """ Holds a substring index for a text T """
    def __init__(self, t, k):
        """ Create index from all substrings of t of length k """
        self.k = k  # k-mer length (k)
        self.index = []
        for i in range(len(t) - k + 1):  # for each k-mer
            self.index.append((t[i:i+k], i))  # add (k-mer, offset) pair
        self.index.sort()  # alphabetize by k-mer

    def query(self, p):
        """ Return index hits for first k-mer of p """
        kmer = p[:self.k]  # query with first k-mer
        i = bisect.bisect_left(self.index, (kmer, -1))  # binary search
        hits = []
        while i < len(self.index):  # collect matching index entries
            if self.index[i][0] != kmer:
                break
            hits.append(self.index[i][1])
            i += 1
        return hits


def query_pigeonhole(p, t, index):
    """
    Finds approximate occurrences of a length-24 pattern P in text T
    with up to 2 mismatches using an 8-mer Index object.
    """
    k = index.k             # 8
    partition_len = 8       # 24 bases / 3 partitions
    all_matches = set()
    total_index_hits = 0    # Tracks total text locations evaluated

    # Slice P into 3 separate 8-mer segments
    # Part 1: p[0:8], Part 2: p[8:16], Part 3: p[16:24]
    for part_idx in range(3):
        start_p = part_idx * partition_len
        end_p = start_p + partition_len
        partition = p[start_p:end_p]

        # Query the index wrapper using the current partition slice
        hits = index.query(partition)
        total_index_hits += len(hits)

        # Verify the flanking downstream and upstream elements for each hit
        for hit in hits:
            # Shift back to align the true structural start index of P in T
            text_start = hit - start_p

            # Boundary check: Ensure full length-24 string fits inside T
            if text_start < 0 or text_start + len(p) > len(t):
                continue

            # Compute substitutions (mismatches)
            mismatches = 0
            for i in range(len(p)):
                if t[text_start + i] != p[i]:
                    mismatches += 1
                    if mismatches > 2:  # Exit calculation early if threshold fails
                        break

            if mismatches <= 2:
                all_matches.add(text_start)

    return sorted(list(all_matches)), total_index_hits


def read_fasta_completely_cleaned(filename):
    """Parses a FASTA file, removing headers and discarding ALL line-break formatting."""
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            if not line.startswith('>'):
                # strip removes outer whitespace; replace removes internal carriage returns/newlines
                clean_line = line.strip().replace('\r', '').replace('\n', '')
                sequence.append(clean_line)
    return ''.join(sequence)

# Swap this line in your execution block:
t = read_fasta_completely_cleaned('chr1.GRCh38.excerpt.fasta')


# --- Automatic Test Execution Wrapper ---
if __name__ == "__main__":
    # 1. Clean parse your local file wrapper
    filename = 'chr1.GRCh38.excerpt.fasta'
    print(f"Reading and cleaning {filename}...")
    t = read_fasta_cleaned(filename)
    print(f"Verified DNA sequence length: {len(t)} bases (Target: 568214)")

    # 2. Initialize the index structure
    print("\nBuilding the 8-mer index tree... (this might take 5-10 seconds)")
    index_8mer = Index(t, k=8)
    print("Index build successful.")

    # 3. Target testing pattern (Exactly 24 bases long)
    # This is the front slice of the Alu sequence from the previous question
    pattern = "GGCGCGGTGGCTCACGCCTGTAAT"
    print(f"\nQuerying pattern: {pattern}")

    # 4. Compute approx matches
    occurrences, hits_evaluated = query_pigeonhole(pattern, t, index_8mer)

    print("\n--- RESULTS ---")
    print(f"Approximate matches found (index starts): {occurrences}")
    print(f"Total occurrences count: {len(occurrences)}")
    print(f"Total candidate index hits filtered and verified: {hits_evaluated}")

In [ ]:
def naive_2mm_forward_only(p, t):
    occurrences = []
    p_len = len(p)
    t_len = len(t)

    for i in range(t_len - p_len + 1):
        mismatches = 0
        match_valid = True

        for j in range(p_len):
            # Comparing directly ONLY to the forward pattern p
            if t[i + j] != p[j]:
                mismatches += 1

            if mismatches > 2:
                match_valid = False
                break

        if match_valid:
            occurrences.append(i)

    return occurrences

human = read_fasta_cleaned('chr1.GRCh38.excerpt.fasta')

# Forward strand only
ans = naive_2mm_forward_only("GGCGCGGTGGCTCACGCCTGTAAT", human)
print("Forward strand only occurrences:", len(ans))


In [ ]:
import bisect

class SubseqIndex(object):
    """ Holds a subsequence index for a text T """

    def __init__(self, t, k, ival):
        """ Create index from all subsequences of t of length k spaced by ival """
        self.k = k  # number of characters in subseq
        self.ival = ival  # spacing between characters
        self.index = []
        self.span = 1 + ival * (k - 1)

        # Build index
        for i in range(len(t) - self.span + 1):
            subseq = t[i:i+self.span:ival]
            self.index.append((subseq, i))
        self.index.sort()

    def query(self, p):
        """ Return index hits for first subseq of p """
        subseq = p[:self.span:self.ival]
        i = bisect.bisect_left(self.index, (subseq, -1))
        hits = []
        while i < len(self.index):
            if self.index[i][0] != subseq:
                break
            hits.append(self.index[i][1])
            i += 1
        return hits


def query_subseq(p, t, index):
    """
    Finds approximate occurrences of P in T with up to 2 mismatches
    using a SubseqIndex built with k=8, ival=3.
    """
    all_matches = set()
    total_index_hits = 0

    # We query for each of the 3 possible subsequence phases (0, 1, 2)
    for i in range(index.ival):
        # Slice the pattern starting at phase i
        subseq_query = p[i:]

        # Query the index
        hits = index.query(subseq_query)
        total_index_hits += len(hits)

        # Verify alignments
        for hit in hits:
            text_start = hit - i
            if text_start < 0 or text_start + len(p) > len(t):
                continue

            mismatches = 0
            for j in range(len(p)):
                if t[text_start + j] != p[j]:
                    mismatches += 1
                    if mismatches > 2:
                        break

            if mismatches <= 2:
                all_matches.add(text_start)

    return sorted(list(all_matches)), total_index_hits


def read_fasta_completely_cleaned(filename):
    """Parses a FASTA file, removing headers and discarding ALL line breaks."""
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            if not line.startswith('>'):
                sequence.append(line.strip().replace('\r', '').replace('\n', ''))
    return ''.join(sequence)

# --- Execution ---
if __name__ == "__main__":
    t = read_fasta_completely_cleaned('chr1.GRCh38.excerpt.fasta')

    # Create SubseqIndex with k=8, ival=3
    subseq_idx = SubseqIndex(t, k=8, ival=3)

    # Target Pattern
    pattern = "GGCGCGGTGGCTCACGCCTGTAAT"

    occurrences, total_hits = query_subseq(pattern, t, subseq_idx)
    print(f"Total Approximate Matches: {len(occurrences)}")
    print(f"Total Subsequence Index Hits Evaluated: {total_hits}")